# PET introduction: list mode and sinograms

This tutorial reconstructs a GATE simulation of an FDG brain scan on a Siemens Biograph mMR. It keeps only the true coincidences, so there are no randoms or scatter to correct for, and shows the two ways PyTomography represents PET data:

- **list mode**, one entry per detected event, giving the pair of crystals that detected it;
- **sinograms**, the same events counted in bins of angle, radial offset and axial plane.

Both are reconstructed with the same object, normalisation and attenuation, so you can compare them directly. The [GATE tutorials](t_PETGATE_scat_sino.ipynb) then add randoms and scatter.

> **Data.** This tutorial reads `PET/GATE-mMR-Brain` (GATE simulation of an FDG brain phantom on a Siemens Biograph mMR, with a normalisation scan) from the folder set by the `PYTOMOGRAPHY_DATA` environment variable. <a href="../tutorials/data.html#data-pet-gate-mmr-brain">Where to get <code>PET/GATE-mMR-Brain</code></a>.

In [ ]:
import os
from pathlib import Path

# Tutorial data: the folder set by the PYTOMOGRAPHY_DATA environment variable (see Tutorial data in the docs)
DATA = Path(os.environ.get("PYTOMOGRAPHY_DATA", "~/pytomography_data")).expanduser()
# Results go here, never into the data folder
OUTPUT = Path(os.environ.get("PYTOMOGRAPHY_OUTPUT", "pytomography_outputs")).expanduser() / "PET/GATE-mMR-Brain"
OUTPUT.mkdir(parents=True, exist_ok=True)

In [ ]:
import time
import torch
import matplotlib.pyplot as plt
import pytomography
from pytomography.metadata import ObjectMeta
from pytomography.metadata.PET import PETLMProjMeta, PETSinogramPolygonProjMeta
from pytomography.projectors.PET import PETLMSystemMatrix, PETSinogramSystemMatrix
from pytomography.likelihoods import PoissonLogLikelihood
from pytomography.algorithms import OSEM
from pytomography.io.PET import gate

## 1. The scanner

GATE describes the scanner in a macro file. `get_detector_info` turns it into a dictionary of crystal and ring counts and positions, which every PET function in PyTomography uses.

In [ ]:
path = DATA / 'PET' / 'GATE-mMR-Brain'
info = gate.get_detector_info(path=os.path.join(path, 'mMR_Geometry.mac'), mean_interaction_depth=9, min_rsector_difference=0)
print(f"{info['NrRings']} rings of {info['NrCrystalsPerRing']} crystals, radius {info['radius']} mm")

## 2. Normalisation

Crystals differ in efficiency, and some lines of response are seen more often than others because of the geometry. A scan of a thin cylindrical shell, which irradiates every line of response almost equally, measures these differences as a weight for every crystal pair. This reads 36 ROOT files (44 GB), so the first run takes a while; the result is cached in `OUTPUT` and shared with the other GATE tutorials.

In [ ]:
weights_file = os.path.join(OUTPUT, 'normalization_weights.pt')
if not os.path.exists(weights_file):
    normalization_paths = [os.path.join(path, f'normalization_scan/mMR_Norm_{i}.root') for i in range(1, 37)]
    normalization_weights = gate.get_normalization_weights_cylinder_calibration(
        normalization_paths, info, cylinder_radius=318, include_randoms=False)  # shell radius in mm
    torch.save(normalization_weights, weights_file)
normalization_weights = torch.load(weights_file)

## 3. The true coincidences

GATE records, for each coincidence, whether its two photons came from the same decay and whether either scattered. Keeping only unscattered photon pairs from the same decay gives the true coincidences. File 24 of the simulation is empty and is skipped.

In [ ]:
ids_file = os.path.join(OUTPUT, 'detector_ids_primary_only.pt')
if not os.path.exists(ids_file):
    paths = [os.path.join(path, f'all_physics/mMR_voxBrain_{i}.root') for i in range(1, 55) if i != 24]
    detector_ids = gate.get_detector_ids_from_root(paths, info, include_randoms=False, include_scatters=False)
    torch.save(detector_ids, ids_file)
detector_ids = torch.load(ids_file)
print(f"{detector_ids.shape[0] / 1e6:.1f} million true coincidences")

## 4. Object space and attenuation

Both reconstructions use the same 128 × 128 × 96 grid of 2 mm voxels. The attenuation map comes from the phantom: `get_attenuation_map_nifti` reads it, aligns it to the grid, and converts it to mm⁻¹.

In [ ]:
object_meta = ObjectMeta(dr=(2, 2, 2), shape=(128, 128, 96))  # mm, voxels
atten_map = gate.get_attenuation_map_nifti(os.path.join(path, 'fdg_pet_phantom_umap.nii.gz'), object_meta)
atten_map = atten_map.to(pytomography.dtype).to(pytomography.device)

## 5. List-mode reconstruction

The list-mode system matrix projects the object onto the line of response of every event. Its sensitivity image, which the algorithm needs, uses the normalisation weights of every possible crystal pair.

In [ ]:
proj_meta_lm = PETLMProjMeta(detector_ids=detector_ids[:, :2], info=info, weights_sensitivity=normalization_weights)
system_matrix_lm = PETLMSystemMatrix(object_meta, proj_meta_lm, N_splits=10, attenuation_map=atten_map)
likelihood_lm = PoissonLogLikelihood(system_matrix_lm)

t0 = time.time()
recon_lm = OSEM(likelihood_lm)(n_iters=4, n_subsets=14)
print(f"list mode: {time.time() - t0:.1f} s")

## 6. Sinogram reconstruction

Binning the same events gives a sinogram whose size depends on the scanner, not on the number of events. The normalisation weights are binned the same way.

In [ ]:
sinogram = gate.listmode_to_sinogram(detector_ids, info)
normalization_sinogram = gate.get_norm_sinogram_from_listmode_data(normalization_weights, info)
print(f"sinogram {tuple(sinogram.shape)}: {sinogram.numel() / 1e6:.0f} million bins for {detector_ids.shape[0] / 1e6:.1f} million events")

proj_meta_sino = PETSinogramPolygonProjMeta(info)
system_matrix_sino = PETSinogramSystemMatrix(
    object_meta, proj_meta_sino,
    sinogram_sensitivity=normalization_sinogram,
    attenuation_map=atten_map,
    N_splits=10,
    device='cpu')  # keep the large sinograms in host memory
likelihood_sino = PoissonLogLikelihood(system_matrix_sino, sinogram)

t0 = time.time()
recon_sino = OSEM(likelihood_sino)(n_iters=4, n_subsets=14)
print(f"sinogram: {time.time() - t0:.1f} s")

## 7. Comparison

The two reconstructions use the same events and the same physics, so they should agree closely. Small differences come from the binning: a sinogram bin groups lines of response that are close but not identical.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(10, 7), constrained_layout=True)
for row, (recon, name) in enumerate([(recon_lm, 'List mode'), (recon_sino, 'Sinogram')]):
    r = recon.cpu()
    vmax = float(r.max()) * 0.8
    for ax, image in zip(axes[row], [r[64].T, r[:, 64].T, r[:, :, 48].T]):
        ax.imshow(image, cmap='Greys', origin='lower', vmax=vmax)
        ax.axis('off')
    axes[row, 0].set_title(name, loc='left')
plt.show()

In [ ]:
difference = (recon_lm - recon_sino).abs().sum() / recon_lm.abs().sum()
print(f"relative difference between the two reconstructions: {100 * difference.item():.1f}%")

## Next steps

- The [GATE sinogram](t_PETGATE_scat_sino.ipynb) and [list-mode](t_PETGATE_scat_lm.ipynb) tutorials reconstruct all events, estimating randoms from the delayed coincidences and scatter with single scatter simulation.
- The time-of-flight versions show how timing information sharpens the reconstruction.